# 🌊 Hero-Mamba: Robust Underwater Image Restoration with Physical Priors
**Paper**: arXiv:2404.16266 — *Hero-Mamba: High-Efficiency and Robust Underwater Image Restoration with State Space Models and Physical Priors*

### 🌟 Key Novelties in Hero-Mamba:
1. **Dual-Domain Stem**: Integrates spatial RGB features with frequency spectral features (FFT amplitude & phase) via SS2D.
2. **Physics-Informed Color Fusion**: Injects a background-light prior ($b_p$) into intermediate features using the optical model $J = I \cdot t + B(1 - t)$.
3. **Multi-Scale Fusion (MS-Fusion)**: Parallel $1\times 1$, $3\times 3$, and $5\times 5$ depthwise separable convolutions.
4. **Composite HeroLoss**: $0.3 \mathcal{L}_1 + 0.8 \mathcal{L}_{ssim} + 0.1 \mathcal{L}_{contrastive}$ with VGG-19 perceptual feature distance.

### ⚙️ Kaggle Settings:
1. **Accelerator**: Select **GPU P100** or **GPU T4 x2** (Settings panel on right 👉).
2. **Internet**: Set to **ON** (needed for `pip install mamba-ssm` and VGG19 weights).
3. **Persistence**: Variables and files in `/kaggle/working/` can be downloaded.

In [ ]:
# Step 1: Install official fused CUDA SSM kernels for Linux
# Uses ninja and --no-build-isolation so pip links against Kaggle's PyTorch & CUDA runtime
!pip install -q ninja packaging
!pip install -q causal-conv1d>=1.4.0 --no-build-isolation
!pip install -q mamba-ssm>=2.0.0 --no-build-isolation

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")

try:
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
    print("\n[SUCCESS] Official fused CUDA mamba_ssm kernel is LOADED! Fast SSM Mamba = True")
except Exception as e:
    print("\n[!] Standard build failed, attempting direct compatibility installation...")
    !pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
    !pip install -q causal-conv1d>=1.4.0 mamba-ssm>=2.0.0 --no-build-isolation
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
    print("\n[SUCCESS] Fused CUDA mamba_ssm kernel successfully installed via fallback!")

### 🏗️ Step 2: Hero-Mamba Model Architecture Definition

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
    HAS_MAMBA = True
except Exception:
    HAS_MAMBA = False

# --------------------------------------------------------------------------- #
# 1. S6 / Selective Scan Core
# --------------------------------------------------------------------------- #
def selective_scan_ref(u, delta, A, B, C, D, delta_bias):
    b, d, l = u.shape
    g = B.shape[1]
    u, delta = u.float(), F.softplus(delta.float() + delta_bias[None, :, None])
    B = B.float().repeat_interleave(d // g, dim=1)
    C = C.float().repeat_interleave(d // g, dim=1)
    x = u.new_zeros(b, d, A.shape[1])
    ys = []
    for i in range(l):
        dt = delta[:, :, i, None]
        x = torch.exp(dt * A) * x + dt * B[:, :, :, i] * u[:, :, i, None]
        ys.append((x * C[:, :, :, i]).sum(-1))
    return torch.stack(ys, -1) + u * D[None, :, None]

def selective_scan(u, delta, A, B, C, D, delta_bias):
    if HAS_MAMBA and u.is_cuda:
        return selective_scan_fn(u, delta, A, B, C, D, z=None,
                                 delta_bias=delta_bias, delta_softplus=True)
    return selective_scan_ref(u, delta, A, B, C, D, delta_bias)

class S6(nn.Module):
    def __init__(self, d, d_state=16, K=1):
        super().__init__()
        self.d, self.N, self.K = d, d_state, K
        self.R = math.ceil(d / 16)
        self.x_proj_weight = nn.Parameter(torch.randn(K, self.R + 2 * d_state, d) * (d ** -0.5))
        self.dt_proj_weight = nn.Parameter(torch.empty(K, d, self.R).uniform_(-self.R ** -0.5, self.R ** -0.5))
        dt = torch.exp(torch.rand(K, d) * (math.log(0.1) - math.log(1e-3)) + math.log(1e-3)).clamp(min=1e-4)
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt)))
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1).float()).repeat(K * d, 1))
        self.D = nn.Parameter(torch.ones(K * d))

    def forward(self, xs):
        b, K, d, L = xs.shape
        x_dbl = torch.einsum("bkdl,kcd->bkcl", xs, self.x_proj_weight)
        dts, Bs, Cs = torch.split(x_dbl, [self.R, self.N, self.N], dim=2)
        dts = torch.einsum("bkrl,kdr->bkdl", dts, self.dt_proj_weight)
        y = selective_scan(
            xs.reshape(b, K * d, L).contiguous(), dts.reshape(b, K * d, L).contiguous(),
            -torch.exp(self.A_log.float()), Bs.contiguous(), Cs.contiguous(),
            self.D.float(), self.dt_bias.reshape(-1).float())
        return y.view(b, K, d, L)

class SS2D(nn.Module):
    """Spatial 4-way selective scan (Linear -> DWConv -> SiLU -> 4-way S6 -> LN -> Gating)."""
    def __init__(self, dim, d_state=16, expand=2):
        super().__init__()
        di = dim * expand
        self.in_proj = nn.Linear(dim, di * 2)
        self.dw = nn.Conv2d(di, di, 3, padding=1, groups=di)
        self.ssm = S6(di, d_state, K=4)
        self.norm = nn.LayerNorm(di)
        self.out_proj = nn.Linear(di, dim)

    def forward(self, x):
        B, C, H, W = x.shape
        L = H * W
        u, z = self.in_proj(x.permute(0, 2, 3, 1)).chunk(2, -1)
        u = F.silu(self.dw(u.permute(0, 3, 1, 2)))
        xh = torch.stack([u.flatten(2), u.transpose(2, 3).flatten(2)], 1)  # (B, 2, D, L)
        xs = torch.cat([xh, xh.flip(-1)], 1)                               # (B, 4, D, L)
        y = self.ssm(xs)
        y = (y[:, 0] + y[:, 2].flip(-1)
             + y[:, 1].reshape(B, -1, W, H).transpose(2, 3).reshape(B, -1, L)
             + y[:, 3].flip(-1).reshape(B, -1, W, H).transpose(2, 3).reshape(B, -1, L))
        y = self.norm(y.view(B, -1, H, W).permute(0, 2, 3, 1))
        y = y * F.silu(z)
        return self.out_proj(y).permute(0, 3, 1, 2)

# --------------------------------------------------------------------------- #
# 2. MS-Fusion Block (Fig. 4, Eqs. 11-12)
# --------------------------------------------------------------------------- #
class MSFusion(nn.Module):
    def __init__(self, dim):
        super().__init__()
        def branch(k):
            p = k // 2
            return nn.Sequential(
                nn.Conv2d(dim, dim, k, padding=p),
                nn.Conv2d(dim, dim, k, padding=p, groups=dim),
                nn.SiLU(),
                nn.Conv2d(dim, dim, k, padding=p),
                nn.Conv2d(dim, dim, k, padding=p, groups=dim))
        self.branches = nn.ModuleList([branch(1), branch(3), branch(5)])
        self.reduce = nn.Conv2d(3 * dim, dim, 1)

    def forward(self, x):
        return self.reduce(torch.cat([b(x) for b in self.branches], 1)) + x

# --------------------------------------------------------------------------- #
# 3. Background-Light Prior (Fig. 2, Eq. 5)
# --------------------------------------------------------------------------- #
class BackgroundLightPrior(nn.Module):
    """
    Estimates global underwater ambient illuminant b_e (via gray-world or FC4),
    applies a learnable offset b_l, and projects to feature dimensions at every scale.
    """
    def __init__(self, chans, fc4=None):
        super().__init__()
        self.fc4 = fc4
        if fc4 is not None:
            for p in fc4.parameters():
                p.requires_grad = False
            fc4.eval()
        self.b_l = nn.Parameter(torch.zeros(1, 3, 1, 1))
        self.convs = nn.ModuleList([nn.Conv2d(3, c, 1) for c in chans])

    @torch.no_grad()
    def estimate(self, img):
        e = self.fc4(img) if self.fc4 is not None else img.mean((2, 3))
        e = e / (e.sum(1, keepdim=True) + 1e-6)  # Normalize illuminant color
        return e.view(-1, 3, 1, 1)

    def forward(self, img, sizes):
        base = self.estimate(img) + self.b_l
        return [conv(base.expand(-1, -1, h, w)) for conv, (h, w) in zip(self.convs, sizes)]

# --------------------------------------------------------------------------- #
# 4. ColorFusion Block (Eqs. 6-10)
# --------------------------------------------------------------------------- #
class ColorFusion(nn.Module):
    """Feature-level optical physical model: c = f * t + B * (1 - t)"""
    def __init__(self, c):
        super().__init__()
        self.to_bf = nn.Conv2d(c, c, 1)
        self.to_t = nn.Conv2d(c, c, 1)
        self.to_w = nn.Conv2d(2 * c, c, 1)

    def forward(self, f, bp):
        bf = torch.sigmoid(self.to_bf(f))
        w = torch.sigmoid(self.to_w(torch.cat([bf, bp], 1)))
        B = w * bf + (1 - w) * bp
        t = torch.sigmoid(self.to_t(f))
        return f * t + B * (1 - t)

# --------------------------------------------------------------------------- #
# 5. Encoder & Decoder Blocks (Eq. 4)
# --------------------------------------------------------------------------- #
class EncoderBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.norm = nn.GroupNorm(1, cin)
        self.ss2d = SS2D(cin)
        self.msf = MSFusion(cin)
        self.down = nn.Conv2d(cin, cout, 2, stride=2)

    def forward(self, x):
        x = x + self.ss2d(self.norm(x))
        return self.down(self.msf(x))

class DecoderBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, padding=1)
        self.bn = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, padding=1)

    def forward(self, x):
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        return F.silu(self.conv2(self.bn(self.conv1(x))))

# --------------------------------------------------------------------------- #
# 6. HeroMamba Network
# --------------------------------------------------------------------------- #
class HeroMamba(nn.Module):
    def __init__(self, fc4=None, dims=(32, 64, 128, 256)):
        super().__init__()
        dec = (dims[2], dims[1], dims[0], dims[0] // 2)  # (128, 64, 32, 16)

        # Dual-domain stem: RGB (3) + FFT spectral amplitude/phase (2) = 5 channels
        self.ss2d_rgb = SS2D(3)
        self.ss2d_fft = SS2D(2)
        self.stem_msf = MSFusion(5)
        self.stem_down = nn.Conv2d(5, dims[0], 3, stride=2, padding=1)

        # Hierarchical Encoder
        self.enc = nn.ModuleList([EncoderBlock(dims[i], dims[i + 1]) for i in range(3)])

        # Background Light & Color Fusion branch
        self.prior = BackgroundLightPrior(dims, fc4)
        self.cf = nn.ModuleList([ColorFusion(c) for c in dims])

        # Decoder with skip connections from ColorFusion
        ins = [dims[3] + dims[3], dec[0] + dims[2], dec[1] + dims[1], dec[2] + dims[0]]
        self.dec = nn.ModuleList([DecoderBlock(i, o) for i, o in zip(ins, dec)])
        self.out = nn.Conv2d(dec[3], 3, 3, padding=1)

    @staticmethod
    def spectral(x):
        # FFT on channel-mean; extract log1p-amplitude and phase angle
        spec = torch.fft.fft2(x.float().mean(1, keepdim=True), norm="ortho")
        return torch.cat([torch.log1p(spec.abs()), spec.angle()], 1)

    def forward(self, x):
        # Dual-domain stem
        i_f = torch.cat([self.ss2d_rgb(x), self.ss2d_fft(self.spectral(x))], 1)
        f = self.stem_down(self.stem_msf(i_f))

        # Encoder path
        feats = [f]
        for blk in self.enc:
            f = blk(f)
            feats.append(f)

        # Physics-informed color fusion
        bps = self.prior(x, [t.shape[-2:] for t in feats])
        cs = [cf(fi, bp) for cf, fi, bp in zip(self.cf, feats, bps)]

        # Decoder path
        d = feats[3]
        for i, blk in enumerate(self.dec):
            d = blk(torch.cat([d, cs[3 - i]], 1))

        return self.out(d) + x  # Global residual connection

print("Hero-Mamba architecture compiled successfully!")

### 🎯 Step 3: Loss Function (HeroLoss: L1 + SSIM + VGG Contrastive Loss)

In [ ]:
import torchvision

def _gauss_window(size=11, sigma=1.5, ch=3, device="cpu"):
    coords = torch.arange(size, device=device, dtype=torch.float32) - size // 2
    g = torch.exp(-(coords ** 2) / (2 * (sigma ** 2)))
    g = g / g.sum()
    w = (g[:, None] * g[None, :])[None, None]
    return w.expand(ch, 1, size, size).contiguous()

def ssim(x, y, win=11, sigma=1.5, C1=0.01 ** 2, C2=0.03 ** 2):
    ch = x.shape[1]
    w = _gauss_window(win, sigma, ch, x.device).to(x.dtype)
    mu1 = F.conv2d(x, w, padding=win // 2, groups=ch)
    mu2 = F.conv2d(y, w, padding=win // 2, groups=ch)
    mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1 * mu2
    sigma1_sq = F.conv2d(x * x, w, padding=win // 2, groups=ch) - mu1_sq
    sigma2_sq = F.conv2d(y * y, w, padding=win // 2, groups=ch) - mu2_sq
    sigma12 = F.conv2d(x * y, w, padding=win // 2, groups=ch) - mu1_mu2
    s = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    return s.mean()

def calc_psnr(pred, gt, max_val=1.0):
    mse = torch.mean((pred - gt) ** 2)
    if mse == 0:
        return float("inf")
    return 20.0 * math.log10(max_val / math.sqrt(mse.item()))

class ContrastiveLoss(nn.Module):
    """Pulls prediction toward GT (positive) and away from raw input (negative) in VGG19 feature space."""
    def __init__(self, pretrained=True):
        super().__init__()
        try:
            weights = torchvision.models.VGG19_Weights.IMAGENET1K_V1 if pretrained else None
            feats = torchvision.models.vgg19(weights=weights).features.eval()
        except Exception:
            feats = torchvision.models.vgg19(pretrained=pretrained).features.eval()
            
        cuts = [(0, 2), (2, 7), (7, 12), (12, 21), (21, 30)]
        self.slices = nn.ModuleList([feats[a:b] for a, b in cuts])
        for p in self.parameters():
            p.requires_grad = False
        self.weights = [1 / 32, 1 / 16, 1 / 8, 1 / 4, 1.0]
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def _feats(self, x):
        x, out = (x - self.mean) / self.std, []
        for s in self.slices:
            x = s(x)
            out.append(x)
        return out

    def forward(self, anchor, pos, neg):
        fa = self._feats(anchor)
        with torch.no_grad():
            fp, fn = self._feats(pos), self._feats(neg)
        return sum(w * F.l1_loss(a, p) / (F.l1_loss(a, n) + 1e-7)
                   for w, a, p, n in zip(self.weights, fa, fp, fn))

class HeroLoss(nn.Module):
    """Paper's Composite Loss: L = 0.3*L1 + 0.8*L_ssim + 0.1*L_contrastive"""
    def __init__(self, alpha=0.3, beta=0.8, gamma=0.1, pretrained_vgg=True):
        super().__init__()
        self.a, self.b, self.g = alpha, beta, gamma
        self.contrast = ContrastiveLoss(pretrained_vgg) if gamma > 0 else None

    def forward(self, pred, gt, raw):
        loss = self.a * F.l1_loss(pred, gt) + self.b * (1 - ssim(pred, gt))
        if self.contrast is not None:
            loss = loss + self.g * self.contrast(pred, gt, raw)
        return loss

print("HeroLoss (L1 + SSIM + Contrastive VGG19) initialized!")

### 📊 Step 4: LSUI Dataset Loader & Augmentation

In [ ]:
import os, glob, random
from pathlib import Path
import pandas as pd
from PIL import Image
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader

class PairDataset(Dataset):
    def __init__(self, root, split="train", splits_csv=None, size=256, max_samples=None):
        self.root = Path(root)
        self.split = split
        self.size = size

        csv_file = Path(splits_csv) if splits_csv else self.root / "lsui_splits.csv"
        df = pd.read_csv(csv_file)
        split_df = df[df["split"].str.lower() == split.lower()]

        self.samples = []
        for _, row in split_df.iterrows():
            fname = str(row["filename"])
            inp_p = self.root / "input" / fname
            gt_p = self.root / "GT" / fname
            if inp_p.exists() and gt_p.exists():
                self.samples.append((str(inp_p), str(gt_p), fname))

        if max_samples and max_samples > 0:
            self.samples = self.samples[:max_samples]
        print(f"Loaded {len(self.samples)} pairs for split '{split}' from {self.root.name}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        inp_path, gt_path, fname = self.samples[i]
        raw = TF.to_tensor(TF.resize(Image.open(inp_path).convert("RGB"), [self.size, self.size]))
        gt = TF.to_tensor(TF.resize(Image.open(gt_path).convert("RGB"), [self.size, self.size]))
        
        # Paired augmentation during training
        if self.split == "train":
            if random.random() < 0.5: raw, gt = raw.flip(-1), gt.flip(-1)
            if random.random() < 0.5: raw, gt = raw.flip(-2), gt.flip(-2)
            
        return raw, gt, fname

### 🚀 Step 5: Training Hero-Mamba on Kaggle GPU ($256 \times 256$, `batch_size = 4`)
Paper settings: AdamW `lr=3e-4`, Cosine Annealing, 250 epochs, batch size 4.

In [ ]:
# Auto-detect dataset directory on Kaggle
DATA_DIR = "/kaggle/input/lsui/LSUI"
if not os.path.exists(DATA_DIR):
    found = glob.glob("/kaggle/input/**/lsui_splits.csv", recursive=True)
    if found:
        DATA_DIR = os.path.dirname(found[0])
        print(f"Auto-detected LSUI dataset at: {DATA_DIR}")
    else:
        print("[!] Please upload or link the LSUI dataset to your Kaggle notebook.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 256
BATCH_SIZE = 4
EPOCHS = 50       # Set to 250 for paper reproduction
LR = 3e-4         # Paper learning rate
CHECKPOINT_DIR = "/kaggle/working/checkpoints_hero"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Model & Loss setup
net = HeroMamba().to(DEVICE)
crit = HeroLoss(alpha=0.3, beta=0.8, gamma=0.1).to(DEVICE)
total_params = sum(p.numel() for p in net.parameters()) / 1e6
print(f"Hero-Mamba Initialized: {total_params:.2f} M parameters")

# Dataloaders
train_loader = DataLoader(PairDataset(DATA_DIR, split="train", size=IMG_SIZE), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(PairDataset(DATA_DIR, split="val", size=IMG_SIZE), batch_size=1, shuffle=False, num_workers=2)

# Optimizer & Scheduler
optimizer = torch.optim.AdamW([p for p in net.parameters() if p.requires_grad], lr=LR, betas=(0.9, 0.999), weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

best_psnr = -1.0
best_path = os.path.join(CHECKPOINT_DIR, "heromamba_best.pth")

print("\n--- Starting Hero-Mamba Training Loop ---")
for epoch in range(1, EPOCHS + 1):
    net.train()
    running_loss, t0 = 0.0, time.time()
    for step, (raw, gt, _) in enumerate(train_loader):
        raw, gt = raw.to(DEVICE), gt.to(DEVICE)
        optimizer.zero_grad()
        pred = net(raw)
        loss = crit(pred, gt, raw)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
        if (step + 1) % 100 == 0 or (step + 1) == len(train_loader):
            print(f"[Epoch {epoch:02d}/{EPOCHS:02d}] Step {step+1:04d}/{len(train_loader):04d} | Loss: {loss.item():.5f} | Avg: {running_loss/(step+1):.5f}")
            
    scheduler.step()
    
    # Validation step
    net.eval()
    v_psnr, v_ssim = 0.0, 0.0
    with torch.no_grad():
        for raw, gt, _ in val_loader:
            raw, gt = raw.to(DEVICE), gt.to(DEVICE)
            out = net(raw).clamp(0.0, 1.0)
            v_psnr += calc_psnr(out, gt)
            v_ssim += ssim(out, gt).item()
            
    v_psnr /= len(val_loader)
    v_ssim /= len(val_loader)
    epoch_time = time.time() - t0
    
    print(f"--> Epoch {epoch:02d} Finished ({epoch_time:.1f}s) | Val PSNR: {v_psnr:.2f} dB | Val SSIM: {v_ssim:.4f}")
    if v_psnr > best_psnr:
        best_psnr = v_psnr
        torch.save(net.state_dict(), best_path)
        print(f"    [★ Best Model Saved] PSNR: {best_psnr:.2f} dB -> {best_path}")

print(f"\nTraining complete! Best weights saved at: {best_path}")

### 🖼️ Step 6: Test Evaluation & Display Visual Restorations

In [ ]:
import matplotlib.pyplot as plt

# Load best weights
net.load_state_dict(torch.load(best_path, map_location=DEVICE))
net.eval()

test_loader = DataLoader(PairDataset(DATA_DIR, split="test", size=IMG_SIZE), batch_size=1, shuffle=False)
print(f"Evaluating on {len(test_loader)} test images...")

test_psnr, test_ssim = 0.0, 0.0
sample_visuals = []

with torch.no_grad():
    for i, (raw, gt, fnames) in enumerate(test_loader):
        raw, gt = raw.to(DEVICE), gt.to(DEVICE)
        out = net(raw).clamp(0.0, 1.0)
        test_psnr += calc_psnr(out, gt)
        test_ssim += ssim(out, gt).item()
        
        if i < 4:
            sample_visuals.append((
                raw[0].cpu().permute(1, 2, 0).numpy(),
                out[0].cpu().permute(1, 2, 0).numpy(),
                gt[0].cpu().permute(1, 2, 0).numpy(),
                fnames[0]
            ))

print(f"\n" + "="*60)
print("HERO-MAMBA FINAL TEST BENCHMARK:")
print(f"Mean PSNR: {test_psnr / len(test_loader):.2f} dB")
print(f"Mean SSIM: {test_ssim / len(test_loader):.4f}")
print("="*60)

# Display Visual Comparisons
fig, axes = plt.subplots(len(sample_visuals), 3, figsize=(12, 4 * len(sample_visuals)))
for idx, (inp, out, gt, name) in enumerate(sample_visuals):
    axes[idx, 0].imshow(inp)
    axes[idx, 0].set_title(f"Raw Input ({name})")
    axes[idx, 0].axis("off")
    
    axes[idx, 1].imshow(out)
    axes[idx, 1].set_title(f"Hero-Mamba Enhanced")
    axes[idx, 1].axis("off")
    
    axes[idx, 2].imshow(gt)
    axes[idx, 2].set_title(f"Ground Truth")
    axes[idx, 2].axis("off")

plt.tight_layout()
plt.show()